In [12]:
import pandas as pd
import numpy as np
import time


NUM_ROWS = 10000 

print(f"Generating {NUM_ROWS} rows of synthetic sensor data...")


data = {
    'Timestamp': [],
    'Pixel_X': [],
    'Pixel_Y': [],
    'Depth_mm': [],
    'Temperature_C': [],
    'ToF_Amplitude': [],
    'Target_Label': []
}


categories = ['Victim', 'Surroundings', 'Heated Object']
current_time = time.time()

for _ in range(NUM_ROWS):
   
    label = np.random.choice(categories, p=[0.4, 0.5, 0.1]) 
    
   
    data['Pixel_X'].append(np.random.randint(0, 640))
    data['Pixel_Y'].append(np.random.randint(0, 480))
    data['Timestamp'].append(current_time + np.random.uniform(0, 10))
    data['Target_Label'].append(label)

    
    if label == 'Victim':
        data['Depth_mm'].append(np.random.normal(2000, 300))      
        data['Temperature_C'].append(np.random.normal(37, 1.0)) 
        data['ToF_Amplitude'].append(np.random.randint(70, 100))  
        
    elif label == 'Surroundings':
        data['Depth_mm'].append(np.random.normal(3500, 800))      
        data['Temperature_C'].append(np.random.normal(22.0, 2.0)) 
        data['ToF_Amplitude'].append(np.random.randint(20, 60))   
        
    elif label == 'Heated Object':
        data['Depth_mm'].append(np.random.normal(2000, 500))      
        data['Temperature_C'].append(np.random.normal(55.0, 5.0))
        data['ToF_Amplitude'].append(np.random.randint(60, 95))   


df = pd.DataFrame(data)


df['Depth_mm'] = df['Depth_mm'].round(1)
df['Temperature_C'] = df['Temperature_C'].round(1)
df['Timestamp'] = df['Timestamp'].round(3)

# Save to your computer
csv_filename = 'fused_sensor_training_data.csv'
df.to_csv(csv_filename, index=False)

print(f"Success! Saved perfectly formatted data to {csv_filename}")

Generating 10000 rows of synthetic sensor data...
Success! Saved perfectly formatted data to fused_sensor_training_data.csv


# Check the file location for the colab enviorment 

In [9]:
import os
print(os.listdir("/content"))


['.config', 'fused_sensor_training_data.csv', 'sample_data']


# Read the existing .csv file

In [13]:
import os
import csv

file_path =os.path.join("/content","fused_sensor_training_data.csv")

with open (file_path,mode ="r" , newline='') as file :
    reader = csv.reader(file)

    for row in reader:
        print(row)


['Timestamp', 'Pixel_X', 'Pixel_Y', 'Depth_mm', 'Temperature_C', 'ToF_Amplitude', 'Target_Label']
['1773863659.221', '166', '321', '2481.9', '38.7', '85', 'Victim']
['1773863658.727', '305', '407', '2820.4', '19.7', '59', 'Surroundings']
['1773863655.459', '64', '83', '1793.6', '37.3', '80', 'Victim']
['1773863660.678', '70', '379', '3100.2', '24.0', '46', 'Surroundings']
['1773863655.496', '617', '265', '1468.3', '45.8', '89', 'Heated Object']
['1773863660.595', '525', '22', '2703.4', '21.0', '47', 'Surroundings']
['1773863655.35', '276', '2', '3288.5', '18.6', '47', 'Surroundings']
['1773863654.886', '351', '358', '3259.7', '21.7', '32', 'Surroundings']
['1773863656.937', '88', '21', '4282.2', '20.1', '53', 'Surroundings']
['1773863659.525', '256', '230', '2066.5', '37.7', '88', 'Victim']
['1773863659.891', '446', '85', '3467.8', '18.3', '31', 'Surroundings']
['1773863655.287', '225', '477', '1790.0', '56.7', '74', 'Heated Object']
['1773863655.837', '259', '138', '2172.8', '37.3', '

In [15]:
# Drop rows where the amplitude (signal strength) is too low
# Adjust the threshold (e.g., 20) based on your specific sensor's specs
df_clean = df[df['ToF_Amplitude'] > 20].copy()

# Handle any missing values (NaNs)
df_clean = df_clean.dropna()

print(f"Original rows: {len(df)}, Cleaned rows: {len(df_clean)}")

Original rows: 10000, Cleaned rows: 9872


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [22]:
import os
from google.colab import files

file_path = "/content/fused_sensor_training_data.csv"

if os.path.exists(file_path):
    print("Downloading from:", file_path)
    print("After download, your browser will show the saved location (Downloads folder).")
    files.download(file_path)
else:
    print("File not found at:", file_path)
    print("Try checking /content with: !ls /content")

After download, your browser will show the saved location (Downloads folder).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [21]:
import os
print(os.getcwd())


/content


# Implementing random forest classifier for train the model

In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# Load your dataset
data = pd.read_csv("/content/fused_sensor_training_data.csv")


# Features: The data the model will learn from (e.g., Depth and position)
X = df_clean[['Pixel_X', 'Pixel_Y', 'Depth_mm', 'ToF_Amplitude']]

# Labels: The target category you want the model to predict
y = df_clean['Target_Label']
# Split 50% train / 50% test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)

# Train model
rf_classifier = RandomForestClassifier(n_estimators=200, random_state=42)
rf_classifier.fit(X_train, y_train)

# this is for prediction
y_pred = rf_classifier.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

# Sample prediction
sample = X_test.iloc[1:3]
prediction = rf_classifier.predict(sample)

sample_dict = sample.iloc[0].to_dict()
print(f"\nSample Reading: {sample_dict}")
print(f"Predicted Victim: {'Present' if prediction[0] == 1 else 'Not Present'}")


Accuracy: 0.93

Classification Report:
                precision    recall  f1-score   support

Heated Object       0.84      0.41      0.55       516
 Surroundings       1.00      1.00      1.00      2416
       Victim       0.87      0.98      0.92      2004

     accuracy                           0.93      4936
    macro avg       0.90      0.80      0.82      4936
 weighted avg       0.93      0.93      0.92      4936


Sample Reading: {'Pixel_X': 281.0, 'Pixel_Y': 40.0, 'Depth_mm': 1637.2, 'ToF_Amplitude': 86.0}
Predicted Victim: Not Present
